In [ ]:
import pandas as pd
from datetime import datetime


def generate_lock_recommendations(df, inactivity_threshold=90):
    """
    Takes a DataFrame and returns users requiring review.
    """

    df = df.copy()

    # Convert LastLogon
    df["LastLogon"] = pd.to_datetime(df["LastLogon"])

    # Calculate inactivity
    today = datetime.today()
    df["DaysInactive"] = (today - df["LastLogon"]).dt.days

    # Remove technical users
    df = df[~df["UserType"].isin(["System", "Communication"])]

    # Remove already locked users
    df = df[df["Locked"] == "No"]

    # Flag inactivity
    df["InactiveFlag"] = df["DaysInactive"] > inactivity_threshold

    # Flag high-risk roles
    df["HighRiskRole"] = df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False)

    # Filter review population
    review_df = df[(df["InactiveFlag"]) | (df["HighRiskRole"])]

    return review_df